In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/product_profiles.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/tfidf_matrix.npz
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/tfidf_vectorizer.pkl
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__output__.json
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/custom.css
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/df_filtered.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__output__.json
/ka

In [2]:
import os
for path, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(path, f))

/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/product_profiles.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/tfidf_matrix.npz
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/tfidf_vectorizer.pkl
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__output__.json
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/custom.css
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/df_filtered.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__output__.json
/ka

In [3]:
import numpy as np
import pandas as pd
import pickle
import scipy.sparse as sp
from surprise import SVD
import warnings
warnings.filterwarnings('ignore')

P1 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/'
P2 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/'
P3 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/'

train            = pd.read_csv(P1 + 'train.csv')
test             = pd.read_csv(P1 + 'test.csv')
product_profiles = pd.read_csv(P3 + 'product_profiles.csv')

with open(P2 + 'svd_model.pkl', 'rb') as f:
    svd = pickle.load(f)

product_ids = product_profiles['ProductId'].tolist()
product_idx = {pid: idx for idx, pid in enumerate(product_ids)}

print(f"Train          : {train.shape}")
print(f"Test           : {test.shape}")
print(f"Product profiles: {product_profiles.shape}")
print(f"SVD loaded     : {type(svd)}")

Train          : (175802, 6)
Test           : (18083, 6)
Product profiles: (17538, 4)
SVD loaded     : <class 'surprise.prediction_algorithms.matrix_factorization.SVD'>


In [4]:
!pip install rank_bm25 -q

from rank_bm25 import BM25Okapi
import time

# Tokenize product content
corpus = product_profiles['content'].fillna('').tolist()
tokenized_corpus = [doc.lower().split() for doc in corpus]

start = time.time()
bm25 = BM25Okapi(tokenized_corpus)
elapsed = time.time() - start

print(f"BM25 index built in {elapsed:.1f} seconds")
print(f"Documents indexed : {len(tokenized_corpus):,}")
print(f"Sample tokens (first product): {tokenized_corpus[0][:10]}")

BM25 index built in 5.8 seconds
Documents indexed : 17,538
Sample tokens (first product): ['so', 'fun', 'to', 'read', 'children', 'will', 'find', 'it', 'entertaining', 'and']


In [5]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_bm25_recommendations(product_id, n=10):
    if product_id not in product_idx:
        return []
    
    idx = product_idx[product_id]
    query_tokens = tokenized_corpus[idx]
    
    # BM25 scores for all products
    scores = bm25.get_scores(query_tokens)
    
    # Exclude the product itself
    scores[idx] = -1
    top_indices = np.argsort(scores)[::-1][:n]
    
    return [(product_ids[i], scores[i]) for i in top_indices]

# Test on a sample product
sample_product = product_profiles['ProductId'].iloc[0]
results = get_bm25_recommendations(sample_product, n=5)

print(f"BM25 recommendations for: {sample_product}")
print(f"\n{'Rank':<6} {'ProductId':<15} {'BM25 Score'}")
print("-" * 35)
for rank, (pid, score) in enumerate(results, 1):
    print(f"{rank:<6} {pid:<15} {score:.4f}")

BM25 recommendations for: 0006641040

Rank   ProductId       BM25 Score
-----------------------------------
1      B005BFJGJU      332.4313
2      B006Y02OZO      332.4313
3      B0057ISDW2      332.4313
4      B0095KATS4      332.4313
5      B006ILRBWU      332.4313


In [6]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

P1 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/'
P2 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/'
P3 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/'

train            = pd.read_csv(P1 + 'train.csv')
test             = pd.read_csv(P1 + 'test.csv')
product_profiles = pd.read_csv(P3 + 'product_profiles.csv')

with open(P2 + 'svd_model.pkl', 'rb') as f:
    svd = pickle.load(f)

product_ids = product_profiles['ProductId'].tolist()
product_idx = {pid: idx for idx, pid in enumerate(product_ids)}

print(f"Train           : {train.shape}")
print(f"Test            : {test.shape}")
print(f"Product profiles: {product_profiles.shape}")

Train           : (175802, 6)
Test            : (18083, 6)
Product profiles: (17538, 4)


In [7]:
from sentence_transformers import SentenceTransformer
import time
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
corpus = product_profiles['content'].fillna('').tolist()

print(f"Encoding {len(corpus):,} product profiles on GPU...")

start = time.time()
embeddings = model.encode(corpus, batch_size=128, show_progress_bar=True, device='cuda')
elapsed = time.time() - start

print(f"\nEncoding complete in {elapsed:.1f} seconds")
print(f"Embeddings shape : {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 17,538 product profiles on GPU...


Batches:   0%|          | 0/138 [00:00<?, ?it/s]


Encoding complete in 81.1 seconds
Embeddings shape : (17538, 384)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

def get_st_recommendations(product_id, n=10):
    if product_id not in product_idx:
        return []
    
    idx = product_idx[product_id]
    sim_scores = cosine_similarity([embeddings[idx]], embeddings).flatten()
    sim_scores[idx] = -1  # exclude itself
    top_indices = np.argsort(sim_scores)[::-1][:n]
    
    return [(product_ids[i], round(sim_scores[i], 4)) for i in top_indices]

# Test on same product we used for TF-IDF
sample_product = product_profiles['ProductId'].iloc[0]
results = get_st_recommendations(sample_product, n=10)

print(f"Sentence Transformer recommendations for: {sample_product}")
print(f"\n{'Rank':<6} {'ProductId':<15} {'Similarity'}")
print("-" * 35)
for rank, (pid, score) in enumerate(results, 1):
    print(f"{rank:<6} {pid:<15} {score:.4f}")

Sentence Transformer recommendations for: 0006641040

Rank   ProductId       Similarity
-----------------------------------
1      B000ILIJRM      0.5252
2      B002AQP3I6      0.4656
3      B0014ET2OQ      0.4655
4      B000G1EO14      0.4648
5      B002AP2OS4      0.4618
6      B00122E6S2      0.4511
7      B000EM6PVW      0.4500
8      B001EO7F8G      0.4484
9      B000V6FU12      0.4448
10     B00168AE2Y      0.4440


In [9]:
def evaluate_st(test_df, train_df, k=10, sample_users=500):
    precisions = []
    recalls = []
    all_recs = set()
    
    sampled = test_df['UserId'].unique()[:sample_users]
    
    for user_id in sampled:
        user_test = test_df[test_df['UserId'] == user_id]
        relevant = set(user_test[user_test['Score'] >= 4]['ProductId'].tolist())
        if not relevant:
            continue
        
        user_train = train_df[train_df['UserId'] == user_id]['ProductId'].tolist()
        if not user_train:
            continue
        
        seed = user_train[-1]
        if seed not in product_idx:
            continue
        
        recs = [pid for pid, _ in get_st_recommendations(seed, n=k)]
        all_recs.update(recs)
        
        hits = len(set(recs) & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))
    
    coverage = len(all_recs) / len(product_ids)
    return np.mean(precisions), np.mean(recalls), coverage

print("Evaluating Sentence Transformers...")
p, r, cov = evaluate_st(test, train, k=10, sample_users=500)

print(f"\nSentence Transformer Results (sample=500 users)")
print(f"================================================")
print(f"Precision@10 : {p:.4f}")
print(f"Recall@10    : {r:.4f}")
print(f"Coverage     : {cov*100:.2f}%")
print(f"\nFull Comparison:")
print(f"{'Model':<25} {'Precision@10':>14} {'Recall@10':>12} {'Coverage':>10}")
print("-" * 63)
print(f"{'TF-IDF (baseline)':<25} {'0.0096':>14} {'0.0252':>12} {'9.41%':>10}")
print(f"{'Sentence Transformers':<25} {p:>14.4f} {r:>12.4f} {cov*100:>9.2f}%")

Evaluating Sentence Transformers...

Sentence Transformer Results (sample=500 users)
Precision@10 : 0.0123
Recall@10    : 0.0250
Coverage     : 17.23%

Full Comparison:
Model                       Precision@10    Recall@10   Coverage
---------------------------------------------------------------
TF-IDF (baseline)                 0.0096       0.0252      9.41%
Sentence Transformers             0.0123       0.0250     17.23%


In [10]:
def hybrid_st_recommend(user_id, seed_product, n=10, alpha=0.6):
    if seed_product not in product_idx:
        return []
    
    idx = product_idx[seed_product]
    sim_scores = cosine_similarity([embeddings[idx]], embeddings).flatten()
    sim_scores[idx] = -1
    top_indices = np.argsort(sim_scores)[::-1][:n*3]
    candidates = [(product_ids[i], sim_scores[i]) for i in top_indices]
    
    hybrid_scores = []
    for pid, cb_score in candidates:
        svd_pred = svd.predict(user_id, pid).est
        svd_norm = (svd_pred - 1) / 4
        hybrid = alpha * svd_norm + (1 - alpha) * cb_score
        hybrid_scores.append((pid, hybrid))
    
    hybrid_scores.sort(key=lambda x: x[1], reverse=True)
    return hybrid_scores[:n]

def evaluate_hybrid_st(test_df, train_df, alpha=0.6, k=10, sample_users=500):
    precisions = []
    recalls = []
    all_recs = set()
    
    sampled = test_df['UserId'].unique()[:sample_users]
    
    for user_id in sampled:
        user_test = test_df[test_df['UserId'] == user_id]
        relevant = set(user_test[user_test['Score'] >= 4]['ProductId'].tolist())
        if not relevant:
            continue
        
        user_train = train_df[train_df['UserId'] == user_id]['ProductId'].tolist()
        if not user_train:
            continue
        
        seed = user_train[-1]
        if seed not in product_idx:
            continue
        
        recs = [pid for pid, _ in hybrid_st_recommend(user_id, seed, n=k, alpha=alpha)]
        all_recs.update(recs)
        
        hits = len(set(recs) & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))
    
    coverage = len(all_recs) / len(product_ids)
    return np.mean(precisions), np.mean(recalls), coverage

print("Evaluating SVD + Sentence Transformers hybrid...")
p, r, cov = evaluate_hybrid_st(test, train, alpha=0.6, k=10, sample_users=500)

print(f"\nFinal Comparison (sample=500 users)")
print(f"{'Model':<30} {'Precision@10':>14} {'Recall@10':>12} {'Coverage':>10}")
print("-" * 68)
print(f"{'TF-IDF (CB baseline)':<30} {'0.0096':>14} {'0.0252':>12} {'9.41%':>10}")
print(f"{'Sentence Transformers (CB)':<30} {'0.0123':>14} {'0.0250':>12} {'17.23%':>10}")
print(f"{'SVD + TF-IDF (Hybrid)':<30} {'0.0203':>14} {'0.0350':>12} {'—':>10}")
print(f"{'SVD + ST (Best Hybrid)':<30} {p:>14.4f} {r:>12.4f} {cov*100:>9.2f}%")

Evaluating SVD + Sentence Transformers hybrid...

Final Comparison (sample=500 users)
Model                            Precision@10    Recall@10   Coverage
--------------------------------------------------------------------
TF-IDF (CB baseline)                   0.0096       0.0252      9.41%
Sentence Transformers (CB)             0.0123       0.0250     17.23%
SVD + TF-IDF (Hybrid)                  0.0203       0.0350          —
SVD + ST (Best Hybrid)                 0.0146       0.0256     16.02%


In [11]:
import numpy as np

np.save('st_embeddings.npy', embeddings)

results_summary = {
    'tfidf_cb'    : {'precision': 0.0096, 'recall': 0.0252, 'coverage': 0.0941},
    'st_cb'       : {'precision': 0.0123, 'recall': 0.0250, 'coverage': 0.1723},
    'svd_tfidf'   : {'precision': 0.0203, 'recall': 0.0350, 'coverage': None},
    'svd_st'      : {'precision': 0.0146, 'recall': 0.0256, 'coverage': 0.1602},
    'best_model'  : 'svd_tfidf',
    'best_alpha'  : 0.6
}

import pickle
with open('results_summary.pkl', 'wb') as f:
    pickle.dump(results_summary, f)

print("Saved st_embeddings.npy")
print("Saved results_summary.pkl")
print("\nPhase 5 complete.")

Saved st_embeddings.npy
Saved results_summary.pkl

Phase 5 complete.
